In [0]:
use sql_catalog.sql_schema;
select * from emp;
DROP TABLE IF EXISTS emp1;
DROP TABLE IF EXISTS emp2;
CREATE TABLE emp2 AS
SELECT *
FROM emp
WHERE empno IN (
    7369,
    7499,
    7566,
    7698,
    7782,
    7839,
    7902
);

select * from emp2;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


In [0]:
show tables;


In [0]:
CREATE TABLE emp1 AS
SELECT * FROM emp;


num_affected_rows,num_inserted_rows


####  2.61.	List	of	emps	of	emp1	who	are	not	found	in	emp2.

In [0]:
select * from emp1
where empno not in (select empno from from emp2);

empno,ename,job,mgr,hiredate,sal,comm,deptno
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30
7876,ADAMS,CLERK,7788,1983-01-12,1100,null,20
7900,JAMES,CLERK,7698,1981-12-03,950,null,30
7934,MILLER,CLERK,7782,1982-01-23,1300,null,10


#### 2.62.Find	the	highest	sal	of	EMP	table.


In [0]:
select max(sal) as max_sal 
from emp;

max_sal
5000


#### 2.64. Find	the	highest	paid	employee	of	sales	department

In [0]:
select max(e.sal) as high_sal, d.dname
from emp e 
join dept d on e.deptno = d.deptno

where d.dname='SALES'
group by d.dname;

high_sal,dname
2850,SALES


In [0]:
select d.dname, max(e.sal) as hig_Sal from emp e join dept d
on e.deptno = d.deptno
where d.dname = 'SALES'
group by d.dname

dname,hig_Sal
SALES,2850


In [0]:
select d.dname, e.sal from emp e join dept d
on e.deptno = d.deptno
where d.dname = 'SALES'
and e.sal = (select max(sal) from emp where deptno = d.deptno)


dname,sal
SALES,2850


In [0]:
-- add employee name
SELECT d.dname, e.ename, e.sal, e.empno
FROM emp e
JOIN dept d
ON e.deptno = d.deptno
WHERE d.dname = 'SALES'
AND e.sal = (
    SELECT MAX(e2.sal)
    FROM emp e2
    JOIN dept d2
    ON e2.deptno = d2.deptno
    WHERE d2.dname = 'SALES'
);

dname,ename,sal,empno
SALES,BLAKE,2850,7698


####  2.65.	List	the	most	recently	hired	emp	of	grade3	belongs	to		location CHICAGO.

In [0]:
--select * from salgrade;
select e.*,d.loc 
from emp e
join dept d on e.deptno = d.deptno
join sql_catalog.sql_schema.salgrade s on e.sal between s.losal and s.hisal 
where d.loc = 'CHICAGO' 
 and s.grade = 3
 and e.hiredate = (
    select max(e2.hiredate)
    from emp e2
    join dept d2 on e2.deptno = d2.deptno
    join SALGRADE s2 on e2.SAL BETWEEN s2.LOSAL AND s2.HISAL
    WHERE s2.GRADE = 3
    and d2.loc = 'CHICAGO'
 );


empno,ename,job,mgr,hiredate,sal,comm,deptno,loc
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30,CHICAGO


In [0]:
SELECT e.empno, e.ename, e.job, e.sal, e.hiredate, d.dname, d.loc, s.grade
FROM Emp e
JOIN Dept d ON e.deptno = d.deptno
JOIN Salgrade s ON e.sal BETWEEN s.losal AND s.hisal
WHERE s.grade = 3
  AND d.loc = 'CHICAGO'
ORDER BY e.hiredate DESC;

empno,ename,job,sal,hiredate,dname,loc,grade
7844,TURNER,SALESMAN,1500,1981-09-08,SALES,CHICAGO,3
7499,ALLEN,SALESMAN,1600,1981-02-20,SALES,CHICAGO,3


#### 2.66.	List	the	employees	who	are	senior	to	most	recently	hired	employee working	under	king

In [0]:
select * from emp
where deptno =20;

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7876,ADAMS,CLERK,7788,1983-01-12,1100,null,20
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


In [0]:
select * from emp
where hiredate < (SELECT MAX(e.HIREDATE)
FROM EMP e
JOIN EMP m
    ON e.MGR = m.EMPNO
WHERE m.ENAME = 'KING');

empno,ename,job,mgr,hiredate,sal,comm,deptno
7369,SMITH,CLERK,7902,1980-12-17,800,null,20
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30


In [0]:
SELECT e.empno, e.ename, e.job, e.hiredate, e.sal, e.deptno
FROM Emp e
WHERE e.hiredate < (
    SELECT hiredate
    FROM Emp
    WHERE mgr = (SELECT empno FROM Emp WHERE ename = 'KING')
    ORDER BY hiredate DESC
    LIMIT 1
)
ORDER BY e.hiredate ASC;

empno,ename,job,hiredate,sal,deptno
7369,SMITH,CLERK,1980-12-17,800,20
7499,ALLEN,SALESMAN,1981-02-20,1600,30
7521,WARD,SALESMAN,1981-02-22,1250,30
7566,JONES,MANAGER,1981-04-02,2975,20
7698,BLAKE,MANAGER,1981-05-01,2850,30


#### 2.67.	List	the	details	of	the	employee	belongs	to	newyork	with	grade	3	to	5 except	‘PRESIDENT’	whose	sal>	the	highest	paid	employee	of	Chicago	in	a group	where	there	is	manager	and	salesman	not	working	under	king

In [0]:
use sql_catalog.sql_schema ;
show tables;

database,tableName,isTemporary
sql_schema,dept,false
sql_schema,emp,false
sql_schema,emp1,false
sql_schema,emp2,false
sql_schema,salgrade,false
,_sqldf,true


In [0]:
select e.ename, d.loc 
from emp e 
join dept d on e.deptno = d.deptno
where d.loc = 'NEW YORK';


ename,loc
CLARK,NEW YORK
KING,NEW YORK
MILLER,NEW YORK


In [0]:
SELECT e.empno, e.ename, e.job, e.sal, e.deptno, d.dname, d.loc, s.grade
FROM Emp e
JOIN Dept d ON e.deptno = d.deptno
JOIN Salgrade s ON e.sal BETWEEN s.losal AND s.hisal
WHERE d.loc = 'NEW YORK'
  AND s.grade BETWEEN 3 AND 5
  AND e.job <> 'PRESIDENT'
  AND e.sal > (
        SELECT MAX(e2.sal)
        FROM Emp e2
        JOIN Dept d2 ON e2.deptno = d2.deptno
        WHERE d2.loc = 'CHICAGO'
      )
  AND e.mgr <> (
        SELECT empno FROM Emp WHERE ename = 'KING'
      )
  AND EXISTS (
        SELECT 1 FROM Emp e3 WHERE e3.deptno = e.deptno AND e3.job = 'MANAGER'
      )
  AND EXISTS (
        SELECT 1 FROM Emp e4 WHERE e4.deptno = e.deptno AND e4.job = 'SALESMAN'
      );

empno,ename,job,sal,deptno,dname,loc,grade


####  2.68.	List	the	details	of	the	senior	employee	belongs	to	1981.

In [0]:
select * from emp 
where hiredate in (select min(hiredate) from emp where year(hiredate) = 1981)

empno,ename,job,mgr,hiredate,sal,comm,deptno
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30


####  2.69.	List	the	employees	who	joined	in	1981	with	the	job	same	as	the	most senior	person	of	the	year	1981.

In [0]:
select * from emp where year(hiredate) = 1981 and job in (select job from emp where year(hiredate) = 1981
order by hiredate asc)

empno,ename,job,mgr,hiredate,sal,comm,deptno
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7698,BLAKE,MANAGER,7839,1981-05-01,2850,null,30
7782,CLARK,MANAGER,7839,1981-06-09,2450,null,10
7839,KING,PRESIDENT,null,1981-11-17,5000,null,10
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30
7900,JAMES,CLERK,7698,1981-12-03,950,null,30
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


#### 2.70. List	the	most	senior	empl	working	under	the	king	and	grade	is	more	than	3.

In [0]:
select e.empno, e.ename, e.job, e.hiredate, e.sal, s.grade from emp e
join salgrade s on e.sal between s.losal and s.hisal
where e.mgr = (select empno from emp where ename = 'KING') 
and s.grade > 3
order by e.hiredate asc;

empno,ename,job,hiredate,sal,grade
7566,JONES,MANAGER,1981-04-02,2975,4
7698,BLAKE,MANAGER,1981-05-01,2850,4
7782,CLARK,MANAGER,1981-06-09,2450,4


####  2.71.	Find	the	total	sal	given	to	the	MGR

In [0]:
SELECT SUM(sal) AS total_mgr_salary
FROM emp
WHERE job = 'MANAGER';

total_mgr_salary
8275


####  2.72.	Find	the	total	annual	sal	to	distribute	job	wise	in	the	year	81.

In [0]:
select job, sum(sal*12) as total_sal 
from emp where year(hiredate) = '1981'
group by job;

job,total_sal
SALESMAN,67200
MANAGER,99300
PRESIDENT,60000
CLERK,11400
ANALYST,36000


#### 2.73. Display	total	sal	employee	belonging	to	grade	3.

In [0]:
select sum(e.sal) as total_sal 
from emp e
join salgrade s on e.sal between s.losal and s.hisal
where s.grade = 3;
 

total_sal
3100


#### 2.74. Display	the	average	salaries	of	all	the	clerks

In [0]:
select avg(sal) as avg_Sal from emp 
where job = 'CLERK'

avg_Sal
1037.5


####  2.75.	 List	the	employee in	dept	20	whose	sal	is	> the	average	sal	Of	dept	10 emps.

In [0]:
select * from emp e
--join dept d on e.deptno = d.deptno
where e.deptno = 20 and e.sal > (select avg(e.sal) as avg_sal from emp e where e.deptno = 10)

empno,ename,job,mgr,hiredate,sal,comm,deptno
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20


In [0]:
SELECT * from emp e join dept d on e.deptno = d.deptno
where d.deptno = 20 and e.sal > (select avg(e.sal) as avg_sal from emp e where e.deptno = 10)

empno,ename,job,mgr,hiredate,sal,comm,deptno,deptno,dname,loc
7566,JONES,MANAGER,7839,1981-04-02,2975,null,20,20,RESEARCH,DALLAS
7788,SCOTT,ANALYST,7566,1982-12-09,3000,null,20,20,RESEARCH,DALLAS
7902,FORD,ANALYST,7566,1981-12-03,3000,null,20,20,RESEARCH,DALLAS
